
## План анализа

В ходе EDA будут выполнены следующие шаги:

1. Загрузка и первичный осмотр данных  
2. Анализ пропусков и типов данных  
3. Анализ целевой переменной  
4. Исследование числовых признаков  
5. Анализ категориальных признаков  
6. Географический анализ  
7. Поиск выбросов и аномалий  
8. Анализ корреляций  
9. Формирование выводов  

Импорт нужных библиотек

In [50]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

Загрузка данных

In [51]:
df_train = pd.read_csv('../data/train.csv', index_col='id')

Выведим первые 5 записей

In [52]:
df_train.head(5)

,timestamp,full_sq,life_sq,floor,max_floor,material,build_year,num_room,kitch_sq,state,...,cafe_count_5000_price_2500,cafe_count_5000_price_4000,cafe_count_5000_price_high,big_church_count_5000,church_count_5000,mosque_count_5000,leisure_count_5000,sport_count_5000,market_count_5000,price_doc
id,,,,,,,,,,,,,,,,,,,,,
1,2011-08-20,43,27.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,...,9,4,0,13,22,1,0,52,4,5850000
2,2011-08-23,34,19.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,...,15,3,0,15,29,1,10,66,14,6000000
3,2011-08-27,43,29.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,...,10,3,0,11,27,0,4,67,10,5700000
4,2011-09-01,89,50.0,9.0,NaN,NaN,NaN,NaN,NaN,NaN,...,11,2,1,4,4,0,0,26,3,13100000
5,2011-09-05,77,77.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,...,319,108,17,135,236,2,91,195,14,16331452


In [53]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 30471 entries, 1 to 30473
Columns: 291 entries, timestamp to price_doc
dtypes: float64(119), int64(156), object(16)
memory usage: 67.9+ MB


In [54]:
print(df_train.duplicated().sum())
df_train[df_train.duplicated()]

10


,timestamp,full_sq,life_sq,floor,max_floor,material,build_year,num_room,kitch_sq,state,...,cafe_count_5000_price_2500,cafe_count_5000_price_4000,cafe_count_5000_price_high,big_church_count_5000,church_count_5000,mosque_count_5000,leisure_count_5000,sport_count_5000,market_count_5000,price_doc
id,,,,,,,,,,,,,,,,,,,,,
3362,2012-08-27,59,NaN,6.0,NaN,NaN,NaN,NaN,NaN,NaN,...,4,2,0,3,15,1,0,24,4,4506800
4331,2012-10-22,61,NaN,18.0,NaN,NaN,NaN,NaN,NaN,NaN,...,11,2,1,5,4,0,1,32,5,8248500
6994,2013-04-03,42,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,...,3,2,0,2,16,1,0,20,4,3444000
8062,2013-05-22,68,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,...,3,2,0,2,16,1,0,20,4,5406690
8656,2013-06-24,40,NaN,12.0,NaN,NaN,NaN,NaN,NaN,NaN,...,1,0,0,4,6,0,0,4,1,4112800
14007,2014-01-22,46,28.0,1.0,9.0,1.0,1968.0,2.0,5.0,NaN,...,10,1,0,13,15,1,1,61,4,3000000
17407,2014-04-15,134,134.0,1.0,1.0,1.0,0.0,3.0,0.0,1.0,...,0,0,0,0,1,0,0,0,0,5798496
26678,2014-12-17,62,NaN,9.0,17.0,1.0,NaN,2.0,1.0,1.0,...,371,141,26,150,249,2,105,203,13,6552000
28364,2015-03-14,62,NaN,2.0,17.0,1.0,NaN,2.0,1.0,1.0,...,371,141,26,150,249,2,105,203,13,6520500


In [70]:
df_train['timestamp'] = pd.to_datetime(df_train['timestamp'])

In [71]:
df_train.select_dtypes(include='object')

,product_type,sub_area,culture_objects_top_25,thermal_power_plant_raion,incineration_raion,oil_chemistry_raion,radiation_raion,railroad_terminal_raion,big_market_raion,nuclear_reactor_raion,detention_facility_raion,water_1line,big_road1_1line,railroad_1line,ecology
id,,,,,,,,,,,,,,,
1,Investment,Bibirevo,no,no,no,no,no,no,no,no,no,no,no,no,good
2,Investment,Nagatinskij Zaton,yes,no,no,no,no,no,no,no,no,no,no,no,excellent
3,Investment,Tekstil'shhiki,no,no,no,no,yes,no,no,no,no,no,no,no,poor
4,Investment,Mitino,no,no,no,no,no,no,no,no,no,no,no,no,good
5,Investment,Basmannoe,no,no,no,no,yes,yes,no,no,no,no,no,yes,excellent
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30469,Investment,Otradnoe,no,no,yes,no,yes,no,no,no,no,no,no,no,good
30470,Investment,Tverskoe,yes,no,no,no,yes,yes,no,no,yes,no,no,no,poor
30471,OwnerOccupier,Poselenie Vnukovskoe,no,no,no,no,no,no,no,no,no,no,no,no,no data


In [55]:
df_train = df_train.drop_duplicates()

In [63]:
numeric_columns = df_train.select_dtypes(include=[np.number]).columns.tolist()

zero_negative_summary = []

for col in numeric_columns:
    zero_count = (df_train[col] == 0).sum()
    neg_count = (df_train[col] < 0).sum()

    zero_negative_summary.append({
        "column": col,
        "zero_count": zero_count,
        "zero_share": zero_count / len(df_train),
        "negative_count": neg_count,
        "negative_share": neg_count / len(df_train),
        "min": df_train[col].min(),
        "median": df_train[col].median(),
        "max": df_train[col].max()
    })

zero_negative_summary = pd.DataFrame(zero_negative_summary).sort_values("zero_count", ascending=False)
zero_negative_summary

,column,zero_count,zero_share,negative_count,negative_share,min,median,max
155,mosque_count_500,30312,0.995108,0,0.0,0.00,0.00,1.000000e+00
178,mosque_count_1000,29877,0.980828,0,0.0,0.00,0.00,1.000000e+00
152,cafe_count_500_price_high,29627,0.972621,0,0.0,0.00,0.00,3.000000e+00
201,mosque_count_1500,29309,0.962181,0,0.0,0.00,0.00,1.000000e+00
175,cafe_count_1000_price_high,29100,0.955320,0,0.0,0.00,0.00,7.000000e+00
...,...,...,...,...,...,...,...,...
28,full_all,0,0.000000,0,0.0,2546.00,85219.00,1.716730e+06
252,prom_part_5000,0,0.000000,0,0.0,0.21,8.98,2.856000e+01
251,green_part_5000,0,0.000000,0,0.0,3.52,19.76,7.546000e+01
32,young_male,0,0.000000,0,0.0,189.00,5470.00,2.097700e+04


In [64]:
zero_negative_summary.describe()

,zero_count,zero_share,negative_count,negative_share,min,median,max
count,275.000000,275.000000,275.0,275.0,2.750000e+02,2.750000e+02,2.750000e+02
mean,6303.047273,0.206922,0.0,0.0,8.022663e+03,7.101290e+04,1.399960e+06
std,9033.692531,0.296566,0.0,0.0,1.256443e+05,7.396747e+05,1.416090e+07
min,0.000000,0.000000,0.0,0.0,0.000000e+00,0.000000e+00,5.218671e-01
25%,0.000000,0.000000,0.0,0.0,0.000000e+00,1.000000e+00,3.350000e+01
50%,678.000000,0.022258,0.0,0.0,0.000000e+00,5.000000e+00,9.100000e+01
75%,10131.500000,0.332606,0.0,0.0,2.919749e-01,6.900000e+01,1.854165e+03
max,30312.000000,0.995108,0.0,0.0,2.081628e+06,1.050803e+07,2.060718e+08


In [84]:
zero_negative_summary_30 = zero_negative_summary.head(30)
fig = px.bar(x=zero_negative_summary_30['zero_count'], y=zero_negative_summary_30['column'], title='Количество нулей по признакам, топ 30')
fig.show()

In [85]:
df_na_sum = df_train.isna().sum().reset_index().rename(columns={0:'NA_sum', 'index':'feature'}).sort_values('NA_sum', ascending=False).head(40)
fig = px.bar(x=df_na_sum['NA_sum'], y=df_na_sum['feature'], title='Количество пропусков по признакам, топ 40')
fig.show()

## Анализ таргета

In [ ]:
print(f'Кол во пропусков у таргета {df_train['price_doc'].isna().sum()}')
print(f"Кол во нулей у таргета {(df_train['price_doc']==0).sum()}")


Кол во пропусков у таргета 0
Кол во нулей у таргета 0


In [80]:
df_train['price_doc'].describe()

count    3.046100e+04
mean     7.123676e+06
std      4.780682e+06
min      1.000000e+05
25%      4.740002e+06
50%      6.275947e+06
75%      8.300000e+06
max      1.111111e+08
Name: price_doc, dtype: float64

In [86]:
fig = px.histogram(x=df_train['price_doc'])
fig.show()

In [87]:
fig = px.histogram(x=np.log(df_train['price_doc']))
fig.show()